# Bloch-Sphere Playground

The Bloch sphere is the standard mental model for a single qubit. By the end of this notebook you will see *why* it is useful, and you will be able to predict measurement outcomes by looking at a point on the sphere.

**Objectives:**
- Parametrize any single-qubit pure state with two angles `theta` and `phi`
- Predict P(|0>) and P(|1>) from the angles alone
- Use an interactive widget to drag the state around and watch probabilities update
- See how gates act geometrically (rotations around the sphere)

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow
np.set_printoptions(precision=3, suppress=True)

## 1. The Bloch parametrization

Any single-qubit pure state can be written, up to an irrelevant global phase, as

$$|\psi\rangle = \cos(\theta/2) |0\rangle + e^{i \phi} \sin(\theta/2) |1\rangle$$

where

- $\theta \in [0, \pi]$ is the **polar angle** (north pole = $|0\rangle$, south pole = $|1\rangle$)
- $\phi \in [0, 2\pi)$ is the **azimuthal angle** (relative phase between the two amplitudes)

Two angles. The whole single-qubit state space lives on a sphere.

In [ ]:
def state_from_bloch(theta, phi):
    """Build |psi> = cos(theta/2)|0> + e^{i phi} sin(theta/2)|1>."""
    return np.array([
        np.cos(theta / 2),
        np.exp(1j * phi) * np.sin(theta / 2),
    ], dtype=complex)

# Famous states
for name, (theta, phi) in [
    ('|0>',  (0,         0)),
    ('|1>',  (np.pi,     0)),
    ('|+>',  (np.pi/2,   0)),
    ('|->',  (np.pi/2,   np.pi)),
    ('|+i>', (np.pi/2,   np.pi/2)),
    ('|-i>', (np.pi/2, 3*np.pi/2)),
]:
    psi = state_from_bloch(theta, phi)
    print(f'{name:>5}  theta={theta:.3f}  phi={phi:.3f}   psi={psi}')

## 2. Measurement probabilities from `theta`

Because the amplitudes are $\cos(\theta/2)$ and $e^{i\phi}\sin(\theta/2)$, the Born rule gives

$$P(0) = \cos^2(\theta/2), \qquad P(1) = \sin^2(\theta/2).$$

The azimuthal angle `phi` does **not** affect computational-basis measurement. It carries the relative phase, which only matters when you apply a gate before measuring.

In [ ]:
thetas = np.linspace(0, np.pi, 200)
p0 = np.cos(thetas / 2) ** 2
p1 = np.sin(thetas / 2) ** 2

plt.plot(thetas, p0, label='P(0) = cos^2(theta/2)', color='#ff9900')
plt.plot(thetas, p1, label='P(1) = sin^2(theta/2)', color='#146eb4')
plt.axvline(np.pi/2, color='gray', linestyle='--', label='equator (theta = pi/2)')
plt.xlabel('theta')
plt.ylabel('probability')
plt.title('Measurement probabilities vs. Bloch polar angle')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Draw the Bloch circle

We will draw the X-Z slice of the Bloch sphere (the page of the screen). The full 3D sphere adds the Y axis to capture `phi`, but the slice is enough to build intuition.

In [ ]:
def plot_bloch_slice(theta, phi=0, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    circle = Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
    ax.add_patch(circle)
    # Project onto the X-Z plane:  x = sin(theta) cos(phi), z = cos(theta)
    x = np.sin(theta) * np.cos(phi)
    z = np.cos(theta)
    ax.annotate(
        '', xy=(x*0.92, z*0.92), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='#ff9900', lw=2),
    )
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.text(0,  1.15, '|0>', ha='center')
    ax.text(0, -1.15, '|1>', ha='center')
    ax.text( 1.15, 0, '|+>', va='center')
    ax.text(-1.15, 0, '|->', va='center')
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
plot_bloch_slice(0,         0,        title='|0>',  ax=axes[0])
plot_bloch_slice(np.pi,     0,        title='|1>',  ax=axes[1])
plot_bloch_slice(np.pi/2,   0,        title='|+>',  ax=axes[2])
plot_bloch_slice(np.pi/2,   np.pi,    title='|->',  ax=axes[3])
plt.tight_layout()
plt.show()

## 4. Interactive playground

Drag the sliders. Watch the arrow move and the probability bars update. This cell uses `ipywidgets` — install with `pip install ipywidgets` if needed.

(If widgets aren't available, the next cell does the same thing with a static grid.)

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    def explore(theta=np.pi/4, phi=0.0):
        psi = state_from_bloch(theta, phi)
        probs = np.abs(psi) ** 2

        fig, (ax_bloch, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.5))
        plot_bloch_slice(theta, phi, ax=ax_bloch, title=f'theta={theta:.2f}, phi={phi:.2f}')

        ax_bar.bar(['|0>', '|1>'], probs, color=['#ff9900', '#146eb4'])
        ax_bar.set_ylim(0, 1)
        ax_bar.set_title('measurement probabilities')
        for i, p in enumerate(probs):
            ax_bar.text(i, p + 0.02, f'{p:.3f}', ha='center')
        plt.tight_layout()
        plt.show()

    interact(
        explore,
        theta=FloatSlider(min=0, max=np.pi,   step=0.05, value=np.pi/4),
        phi  =FloatSlider(min=0, max=2*np.pi, step=0.05, value=0.0),
    )
except ImportError:
    print('ipywidgets not installed — see the static grid below instead.')

In [ ]:
# Static fallback: a grid of states across the Bloch slice.
thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
fig, axes = plt.subplots(1, len(thetas), figsize=(15, 3.5))
for ax, theta in zip(axes, thetas):
    psi = state_from_bloch(theta, 0)
    probs = np.abs(psi) ** 2
    plot_bloch_slice(theta, 0, ax=ax,
                     title=f'theta={theta:.2f}\nP(0)={probs[0]:.2f}, P(1)={probs[1]:.2f}')
plt.tight_layout()
plt.show()

## 5. Gates as rotations

Every single-qubit gate is a **rotation** of the Bloch sphere. For example:

- $X$ rotates 180° about the X-axis — sends $|0\rangle$ to $|1\rangle$
- $Z$ rotates 180° about the Z-axis — sends $|+\rangle$ to $|-\rangle$
- $H$ rotates 180° about the $(X+Z)/\sqrt{2}$ axis — sends $|0\rangle$ to $|+\rangle$
- $R_y(\alpha)$ rotates by $\alpha$ about the Y-axis — useful for state preparation

Watch what happens when we apply gates and re-extract the Bloch angles.

In [ ]:
def bloch_from_state(psi):
    """Recover (theta, phi) from a state vector (up to global phase)."""
    a, b = psi
    # Strip a global phase so a is real and non-negative.
    if abs(a) > 1e-12:
        phase = a / abs(a)
        a, b = a / phase, b / phase
    theta = 2 * np.arctan2(abs(b), abs(a))
    phi = np.angle(b) if abs(b) > 1e-12 else 0.0
    return theta, phi

H = (1/np.sqrt(2)) * np.array([[1,  1], [1, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)

zero = np.array([1, 0], dtype=complex)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (name, psi) in zip(axes, [('|0>', zero), ('X|0>', X @ zero), ('H|0>', H @ zero)]):
    theta, phi = bloch_from_state(psi)
    plot_bloch_slice(theta, phi, ax=ax, title=f'{name}\ntheta={theta:.2f}, phi={phi:.2f}')
plt.tight_layout()
plt.show()

## 6. Self-check

These three questions are the gate. If you can answer them without consulting the cells above, you are ready for `00-foundations`.

In [ ]:
# Q1: What are theta and phi for the state |+i> = (|0> + i|1>) / sqrt(2)?
# Where does it live on the Bloch sphere?
# TODO


In [ ]:
# Q2: A qubit has theta = pi/3. What is P(0)? (Should be in your head, not on the screen.)
# Confirm with code.
# TODO


In [ ]:
# Q3: Apply H to |1>. Use bloch_from_state to recover theta and phi.
# Where is H|1> on the Bloch sphere?
# TODO


### Solutions

In [ ]:
# --- Q1 ---
psi_iplus = np.array([1, 1j]) / np.sqrt(2)
print(bloch_from_state(psi_iplus))      # (pi/2, pi/2)
# Lives on the equator, +Y direction.

# --- Q2 ---
theta = np.pi/3
print('P(0) =', np.cos(theta/2)**2)     # cos^2(pi/6) = 3/4

# --- Q3 ---
one = np.array([0, 1], dtype=complex)
psi = H @ one
print(bloch_from_state(psi))            # (pi/2, pi)  — that's |->

## You finished the prerequisites module.

If the self-checks across all six notebooks felt routine, head to [`../../00-foundations/GUIDE.md`](../../00-foundations/GUIDE.md).

If three or more questions tripped you up, replay the corresponding notebook before continuing. The foundations module assumes everything here is automatic.